# Data Cleaning — Supply Chain Orders
A complete data cleaning workflow applied to `aws_supply_chain_orders_raw.csv`.

**Steps:**
1. Load & First Look
2. Rename & Inspect Columns
3. Fix Data Types
4. Handle Missing Values
5. Detect & Handle Duplicates
6. Outlier Detection & Treatment
7. String Standardisation
8. Feature Engineering
9. Validate & Save Cleaned Data

## 1. Load & First Look

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style='whitegrid')

df = pd.read_csv('aws_supply_chain_orders_raw.csv')
print('Shape:', df.shape)
df.head()

In [ ]:
# Quick summary of data types, non-null counts, memory
df.info()

In [ ]:
# Statistical summary for numeric columns
df.describe()

In [ ]:
# Statistical summary for object (categorical) columns
df.describe(include='object')

## 2. Rename & Inspect Columns

In [ ]:
# Check for accidental whitespace in column names
print('Column names:', df.columns.tolist())
print('Whitespace issues:', [c for c in df.columns if c != c.strip()])

In [ ]:
# Strip whitespace from column names (safe to always run)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print('Cleaned column names:', df.columns.tolist())

## 3. Fix Data Types

In [ ]:
# order_date and delivery_date are stored as strings — convert to datetime
df['order_date']    = pd.to_datetime(df['order_date'],    errors='coerce')
df['delivery_date'] = pd.to_datetime(df['delivery_date'], errors='coerce')

# Verify
print(df[['order_date', 'delivery_date']].dtypes)
df[['order_date', 'delivery_date']].head(3)

In [ ]:
# Convert low-cardinality string columns to Categorical to save memory
cat_cols = ['warehouse', 'region', 'product', 'status']
for col in cat_cols:
    df[col] = df[col].astype('category')

df.dtypes

## 4. Handle Missing Values

In [ ]:
# --- 4a. Count & visualise missing values ---
missing = df.isnull().sum().rename('count')
missing_pct = (df.isnull().mean() * 100).rename('pct')
missing_df = pd.concat([missing, missing_pct], axis=1)
print(missing_df[missing_df['count'] > 0])

In [ ]:
# Heatmap of missing values (useful for larger datasets)
plt.figure(figsize=(10, 4))
sns.heatmap(df.isnull(), cbar=False, yticklabels=False, cmap='YlOrRd')
plt.title('Missing Value Heatmap (yellow = missing)')
plt.show()

In [ ]:
# --- 4b. Inspect rows with missing region ---
print('Rows missing region:', df['region'].isna().sum())
df[df['region'].isna()]

In [ ]:
# Strategy: fill region with 'Unknown' (not enough info to infer it)
df['region'] = df['region'].cat.add_categories('Unknown').fillna('Unknown')
print('Remaining region nulls:', df['region'].isna().sum())

In [ ]:
# --- 4c. Inspect rows with missing delivery_time_days ---
print('Rows missing delivery_time_days:', df['delivery_time_days'].isna().sum())
df[df['delivery_time_days'].isna()]

In [ ]:
# Strategy 1: compute from dates where both dates exist
mask_missing = df['delivery_time_days'].isna()
can_compute  = mask_missing & df['delivery_date'].notna() & df['order_date'].notna()

df.loc[can_compute, 'delivery_time_days'] = (
    (df.loc[can_compute, 'delivery_date'] - df.loc[can_compute, 'order_date'])
    .dt.days
)
print('Still missing after date-imputation:', df['delivery_time_days'].isna().sum())

In [ ]:
# Strategy 2: fill remaining with median per product (group-level imputation)
df['delivery_time_days'] = df.groupby('product', observed=True)['delivery_time_days'] \
    .transform(lambda x: x.fillna(x.median()))

# Final fallback: global median
global_median = df['delivery_time_days'].median()
df['delivery_time_days'] = df['delivery_time_days'].fillna(global_median)

print('Missing after full imputation:', df['delivery_time_days'].isna().sum())

In [ ]:
# Confirm: zero missing values remaining
df.isnull().sum()

## 5. Detect & Handle Duplicates

In [ ]:
# Check for fully duplicated rows
n_dups = df.duplicated().sum()
print(f'Fully duplicated rows: {n_dups}')

In [ ]:
# Check for duplicate order_ids (should be unique business key)
dup_ids = df[df.duplicated(subset='order_id', keep=False)]
print(f'Duplicate order_ids: {dup_ids.shape[0]}')
dup_ids.head()

In [ ]:
# Drop fully duplicated rows (keep first occurrence)
df = df.drop_duplicates(keep='first').reset_index(drop=True)
print('Shape after dropping duplicates:', df.shape)

## 6. Outlier Detection & Treatment

In [ ]:
# --- 6a. Visualise distributions to spot outliers ---
num_cols = ['order_qty', 'delivery_time_days']

fig, axes = plt.subplots(2, len(num_cols), figsize=(12, 8))
for i, col in enumerate(num_cols):
    sns.boxplot(data=df, y=col, ax=axes[0, i])
    axes[0, i].set_title(f'Boxplot — {col}')
    sns.histplot(data=df, x=col, kde=True, ax=axes[1, i])
    axes[1, i].set_title(f'Histogram — {col}')
plt.tight_layout()
plt.show()

In [ ]:
# --- 6b. IQR method to identify outlier boundaries ---
def iqr_bounds(series):
    Q1, Q3 = series.quantile([0.25, 0.75])
    IQR = Q3 - Q1
    return Q1 - 1.5 * IQR, Q3 + 1.5 * IQR

for col in num_cols:
    lo, hi = iqr_bounds(df[col])
    n_out = ((df[col] < lo) | (df[col] > hi)).sum()
    print(f'{col}: lower={lo:.1f}, upper={hi:.1f}, outliers={n_out}')

In [ ]:
# --- 6c. Z-score method ---
from scipy import stats

for col in num_cols:
    z = np.abs(stats.zscore(df[col]))
    print(f'{col}: rows with |z| > 3 = {(z > 3).sum()}')

In [ ]:
# --- 6d. Cap outliers at IQR bounds (Winsorization) ---
# Preferred over dropping: keeps row count, reduces extreme influence
for col in num_cols:
    lo, hi = iqr_bounds(df[col])
    df[col] = df[col].clip(lower=lo, upper=hi)

print('Outliers capped. New ranges:')
df[num_cols].agg(['min', 'max'])

## 7. String Standardisation

In [ ]:
# Check for whitespace or case inconsistencies in string columns
str_cols = ['warehouse', 'region', 'product', 'status', 'order_id']
for col in str_cols:
    unique_vals = df[col].astype(str).unique()
    print(f'{col}: {sorted(unique_vals)}')

In [ ]:
# Strip whitespace and title-case categorical columns
for col in ['warehouse', 'region', 'product', 'status']:
    df[col] = df[col].astype(str).str.strip().str.title().astype('category')

# Strip whitespace from order_id
df['order_id'] = df['order_id'].str.strip()

# Verify
df['status'].value_counts()

## 8. Feature Engineering
Create new columns that are useful for analysis.

In [ ]:
# 8a. Temporal features from order_date
df['order_year']    = df['order_date'].dt.year
df['order_month']   = df['order_date'].dt.month
df['order_month_name'] = df['order_date'].dt.month_name()
df['order_dow']     = df['order_date'].dt.day_name()   # day of week
df['order_quarter'] = df['order_date'].dt.quarter
df[['order_date', 'order_year', 'order_month', 'order_dow', 'order_quarter']].head()

In [ ]:
# 8b. Recalculate delivery_time_days cleanly from dates
df['computed_delivery_days'] = (df['delivery_date'] - df['order_date']).dt.days
df[['order_date', 'delivery_date', 'delivery_time_days', 'computed_delivery_days']].head(5)

In [ ]:
# 8c. Order size buckets (binning a continuous variable)
df['order_size'] = pd.cut(
    df['order_qty'],
    bins=[0, 15, 30, 45, 100],
    labels=['Small', 'Medium', 'Large', 'XLarge']
)
df['order_size'].value_counts().sort_index()

In [ ]:
# 8d. Binary flag: is the order late? (delivery > 10 days = late)
LATE_THRESHOLD = 10
df['is_late'] = (df['delivery_time_days'] > LATE_THRESHOLD).astype(int)
print('Late order rate:', df['is_late'].mean().round(3))
df['is_late'].value_counts()

In [ ]:
# 8e. Revenue proxy: order_qty * product price (fictional prices)
price_map = {'Router': 150, 'Laptop': 900, 'Switch': 300, 'Server': 2500, 'Storage': 600}
df['unit_price']  = df['product'].map(price_map)
df['order_value'] = df['order_qty'] * df['unit_price']
df[['product', 'order_qty', 'unit_price', 'order_value']].head()

## 9. Validate & Save Cleaned Data

In [ ]:
# --- 9a. Final sanity checks ---
print('Shape:', df.shape)
print()
print('Missing values:')
print(df.isnull().sum())
print()
print('Data types:')
print(df.dtypes)

In [ ]:
# --- 9b. Business rule assertions ---

# delivery_date should be >= order_date
bad_dates = df[df['delivery_date'] < df['order_date']]
print(f'Orders where delivery_date < order_date: {len(bad_dates)}')

# order_qty must be positive
print(f'Negative or zero qty rows: {(df["order_qty"] <= 0).sum()}')

# delivery_time_days must be positive
print(f'Negative delivery days: {(df["delivery_time_days"] < 0).sum()}')

# order_id should be unique
print(f'Duplicate order_ids: {df["order_id"].duplicated().sum()}')

In [ ]:
# --- 9c. Final view of cleaned DataFrame ---
df.head(10)

In [ ]:
# --- 9d. Save cleaned data ---
output_path = 'supply_chain_cleaned_v2.csv'
df.to_csv(output_path, index=False)
print(f'Saved {df.shape[0]} rows to {output_path}')

## Summary of Changes Made

| Step | Issue Found | Fix Applied |
|---|---|---|
| Column names | May have whitespace | `str.strip().str.lower()` |
| `order_date`, `delivery_date` | Stored as `object` strings | Converted to `datetime64` |
| Categorical cols | `object` dtype, wasting memory | Converted to `category` dtype |
| `region` | 5 null values | Filled with `'Unknown'` |
| `delivery_time_days` | 11 null values | 1st: computed from dates; 2nd: group median imputation |
| Duplicates | Checked for full-row + `order_id` dupes | Dropped full-row duplicates |
| Outliers | IQR & Z-score check | Winsorization (capped at IQR bounds) |
| String values | Possible inconsistency | `.str.strip().str.title()` normalisation |
| Feature engineering | New useful columns | `order_month`, `order_dow`, `order_quarter`, `order_size`, `is_late`, `order_value` |